# LSEG Data Pull - Macaulay Duration

## 0. Setup

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import lseg.data as ld

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 120)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
CACHE_DATA_DIR = BASE_DIR / "cache"
CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)


# Safety guard: never overwrite euro500.parquet from this notebook.
if not hasattr(pd.DataFrame, "_orig_to_parquet_macaulay"):
    pd.DataFrame._orig_to_parquet_macaulay = pd.DataFrame.to_parquet

def _guarded_to_parquet(self, path, *args, **kwargs):
    try:
        target = Path(path).expanduser().resolve()
    except Exception:
        target = Path(str(path))
    try:
        forbidden = (DATA_DIR / "euro500.parquet").expanduser().resolve()
    except Exception:
        forbidden = DATA_DIR / "euro500.parquet"
    if str(target) == str(forbidden):
        raise RuntimeError(
            "Write blocked: euro500.parquet is read-only in LSEG_DataPull_Macaulay.ipynb"
        )
    return pd.DataFrame._orig_to_parquet_macaulay(self, path, *args, **kwargs)

if pd.DataFrame.to_parquet is not _guarded_to_parquet:
    pd.DataFrame.to_parquet = _guarded_to_parquet

import hashlib
import json
import random
import re
import time
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]




## 1. Input-Parquets laden

In [2]:
EURO500_PATH = DATA_DIR / "euro500.parquet"
EURO500_RETURNS_PATH = DATA_DIR / "euro500_index_returns.parquet"
DAILY_RETURNS_IN_INDEX_PATH = DATA_DIR / "euro500_daily_returns.parquet"

for p in [EURO500_PATH, EURO500_RETURNS_PATH, DAILY_RETURNS_IN_INDEX_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

euro500 = pd.read_parquet(EURO500_PATH)
euro500_returns = pd.read_parquet(EURO500_RETURNS_PATH)
daily_returns_euro500_in_index = pd.read_parquet(DAILY_RETURNS_IN_INDEX_PATH)

print("Loaded:")
print("- euro500:", euro500.shape)
print("- euro500_returns:", euro500_returns.shape)
print("- daily_returns_euro500_in_index:", daily_returns_euro500_in_index.shape)

Loaded:
- euro500: (56500, 25)
- euro500_returns: (7016, 8)
- daily_returns_euro500_in_index: (3457796, 10)


## 2. Shares Outstanding

Fehlende Werte in `shares_outstanding` werden imputiert mit:

$$
\text{shares\_outstanding} = \frac{\text{mcap\_eur}}{\text{PriceClose}}
$$



In [3]:
# Step 2 — Missing shares_outstanding mit mcap_eur / PriceClose auffuellen
required_cols = ['shares_outstanding', 'mcap_eur', 'PriceClose']
missing_required = [c for c in required_cols if c not in euro500.columns]
if missing_required:
    raise KeyError(f"Missing required columns for Step 2: {missing_required}")

for c in required_cols:
    euro500[c] = pd.to_numeric(euro500[c], errors='coerce')

n_total = len(euro500)
missing_before = int(euro500['shares_outstanding'].isna().sum())
coverage_before = 1.0 - (missing_before / n_total) if n_total else np.nan

# Nur dort imputen, wo shares_outstanding fehlt und Divisor valide ist.
fill_mask = (
    euro500['shares_outstanding'].isna()
    & euro500['mcap_eur'].notna()
    & euro500['PriceClose'].notna()
    & (euro500['PriceClose'] != 0)
)

euro500.loc[fill_mask, 'shares_outstanding'] = (
    euro500.loc[fill_mask, 'mcap_eur'] / euro500.loc[fill_mask, 'PriceClose']
)

missing_after = int(euro500['shares_outstanding'].isna().sum())
coverage_after = 1.0 - (missing_after / n_total) if n_total else np.nan
filled_rows = int(fill_mask.sum())

print('Step 2 coverage update (shares_outstanding):')
print(f'- total rows: {n_total}')
print(f'- missing before: {missing_before} ({(1-coverage_before)*100:.2f}%)')
print(f'- missing after : {missing_after} ({(1-coverage_after)*100:.2f}%)')
print(f'- filled rows   : {filled_rows}')
print(f'- coverage before: {coverage_before*100:.2f}%')
print(f'- coverage after : {coverage_after*100:.2f}%')


# Persist Step 2 base panel for all subsequent steps (single-output design)
if "EURO500_EPS_PATH" not in globals():
    EURO500_EPS_PATH = DATA_DIR / "euro500_macaulay.parquet"
euro500.to_parquet(EURO500_EPS_PATH, index=False)
print("Saved Step 2 base panel:", EURO500_EPS_PATH, "rows:", len(euro500))



Step 2 coverage update (shares_outstanding):
- total rows: 56500
- missing before: 18025 (31.90%)
- missing after : 368 (0.65%)
- filled rows   : 17657
- coverage before: 68.10%
- coverage after : 99.35%
Saved Step 2 base panel: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_macaulay.parquet rows: 56500


## 3. Cash EPS Forecasts + Act Value ziehen

In [ ]:
from typing import Dict, List, Optional, Tuple
import os
import concurrent.futures as _cf
# ------------------------------------------------------------
# Step 3 — Cash EPS Forecasts FY1-FY5 + Current Act Value
# Cache: per company (firm_id), extensible across index changes
# Fixes vs. old code: no ISIN:-prefix, rate-limit backoff not abort
# ------------------------------------------------------------


def _asof_from_euro500(df: pd.DataFrame) -> pd.Series:
    if "date" not in df.columns:
        raise ValueError("Missing required date column in euro500.")
    return pd.to_datetime(df["date"], errors="coerce").dt.normalize()

# ---- Config ------------------------------------------------
HORIZONS          = ["FY1", "FY2", "FY3", "FY4", "FY5"]
FY_FIELDS_MAP_S3   = {f"EPS_{h}": f"TR.CashEPSMean(period={h})" for h in HORIZONS}
ACT_FIELD_MAP_S3   = {"EPS_CashAct": "TR.EPSCashActValue"}
EPS_FIELDS_MAP     = {**FY_FIELDS_MAP_S3, **ACT_FIELD_MAP_S3}
TARGET_COLS_S3     = list(EPS_FIELDS_MAP.keys())
REQUIRED_COLS_S3   = list(FY_FIELDS_MAP_S3.keys())

BATCH_SIZE_S3                = 100
FORCE_REFRESH_S3             = False  # If True, still prefill from store/cache; network pulls can overwrite stale values.
CACHE_ONLY_S3                = False
# Master switch: True => Step 3 never pulls from LSEG network, only local store/cache.
DISABLE_LSEG_PULL_S3         = False
MAX_RETRIES_S3               = 1
S3_MINIMIZE_REQUESTS         = True
S3_SPLIT_MAX_DEPTH_S3        = 10
DEBUG_VERBOSE_S3             = False
S3_COMPACT_LOG               = True
S3_COMPACT_OUTPUT            = True
S3_SHOW_FILE_LOGS            = False
S3_DATE_BATCH_SIZE           = 4
RESUME_FROM_LAST_FILLED_S3   = True
INCLUDE_LAST_FILLED_S3      = False   # True => recompute the last filled date as well

# Effective offline mode (also respects existing CACHE_ONLY_S3 flag).
CACHE_ONLY_S3                = bool(CACHE_ONLY_S3 or DISABLE_LSEG_PULL_S3)

RATE_LIMIT_COOLDOWN_SEC_S3   = 5.0
RATE_LIMIT_MULTIPLIER_S3     = 2.0
RATE_LIMIT_HARD_PAUSE_SEC_S3 = 30.0
FAIL_FAST_ON_RATE_LIMIT_S3   = False   # allow retry on 429 (not abort)
S3_CALL_TIMEOUT_SEC          = 45.0    # hard timeout per ld.get_data call to avoid long silent hangs

S3_DP_PROXY_BASE_URL         = "http://127.0.0.1:9005"  # Workspace proxy port override to avoid wrong 9000 fallback.

AUTO_DISABLE_NETWORK_ON_PROTOCOL_ERR_S3 = True

_s3_disable_network_runtime = False
_s3_disable_network_reason = ""
_s3_disable_network_warned = False

S3_STOP_AFTER_R1_WHEN_NO_NETWORK = True
S3_FAIL_FAST_IF_NO_NETWORK_AND_NO_CACHE = True

S3_USE_NODATA_QUARTER_LOG = True

if S3_MINIMIZE_REQUESTS:
    MAX_RETRIES_S3 = 0
    S3_SPLIT_MAX_DEPTH_S3 = 0
S3_MIN_ASOF_DATE            = pd.Timestamp("1998-12-31")

S3_APPEND_TODAY_ASOF         = True

CACHE_VERSION_S3 = "v4_cash_eps_mean_act"
EURO500_EPS_PATH = DATA_DIR / "euro500_macaulay.parquet"
S3_ROWS_PATH     = CACHE_DATA_DIR / "cash_eps_act_step3_rows_v4.parquet"
S3_CKPT_PATH     = CACHE_DATA_DIR / "cash_eps_act_step3_checkpoint_v4.json"
S3_NODATA_Q_PATH  = CACHE_DATA_DIR / "cash_eps_act_step3_no_data_quarters_v1.json"
S3_NODATA_Q_PATH_LEGACY = CACHE_DATA_DIR / "cash_eps_act_step3_no_data_quarters.json"
S3_BAD_IDS_PATH   = CACHE_DATA_DIR / "cash_eps_act_step3_bad_ids_v1.json"
S3_BAD_IDS_PATH_LEGACY = CACHE_DATA_DIR / "cash_eps_act_step3_bad_ids.json"
S3_ROWS_PATH_LEGACY = CACHE_DATA_DIR / "cash_eps_act_step3_rows.parquet"
S3_CKPT_PATH_LEGACY = CACHE_DATA_DIR / "cash_eps_act_step3_checkpoint.json"
S3_CACHE_DIR     = CACHE_DATA_DIR / "cash_eps_act_cache_by_company"
S3_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Suppress LSEG/pandas incompatibility: LSEG internals do `if df:` in __del__ callbacks,
# which raises ValueError in pandas 2.x. These are unraisable exceptions that bypass
# try/except and show as cell errors in Python 3.12+/Jupyter.
import sys as _sys
_s3_prev_unraisablehook = _sys.unraisablehook
def _s3_unraisable_hook(args):
    # LSEG destructors may trigger pandas bool(ValueError) in unraisable context.
    try:
        msg = str(getattr(args, "exc_value", "")).lower()
    except Exception:
        msg = ""
    if "truth value" in msg and "ambiguous" in msg and ("series" in msg or "dataframe" in msg):
        return  # suppress known non-fatal pandas truth-value errors
    _s3_prev_unraisablehook(args)
_sys.unraisablehook = _s3_unraisable_hook

# pandas 2.x + Python 3.13 compatibility: some third-party code does `if df:`
# (or `if series:`), which now raises ValueError. Map truthiness to not-empty.
def _s3_ndframe_bool(self):
    try:
        return not self.empty
    except Exception:
        return True

try:
    _s3_orig_series_bool = pd.Series.__bool__
except Exception:
    _s3_orig_series_bool = None
try:
    _s3_orig_series_nonzero = pd.Series.__nonzero__
except Exception:
    _s3_orig_series_nonzero = None
try:
    _s3_orig_df_bool = pd.DataFrame.__bool__
except Exception:
    _s3_orig_df_bool = None
try:
    _s3_orig_df_nonzero = pd.DataFrame.__nonzero__
except Exception:
    _s3_orig_df_nonzero = None

pd.Series.__bool__ = _s3_ndframe_bool
pd.Series.__nonzero__ = _s3_ndframe_bool
pd.DataFrame.__bool__ = _s3_ndframe_bool
pd.DataFrame.__nonzero__ = _s3_ndframe_bool

# ---- Basic helpers -----------------------------------------
def _s3_asof(df: pd.DataFrame) -> pd.Series:
    if "date" not in df.columns:
        raise ValueError("Missing required date column for Step 3.")
    return pd.to_datetime(df["date"], errors="coerce").dt.normalize()

def _s3_clean(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    return x.where(x.notna() & (x != ""), pd.NA)

def _s3_norm_isin(v) -> Optional[str]:
    if pd.isna(v):
        return None
    s = str(v).strip()
    return (s.split(":", 1)[1].strip() if s.upper().startswith("ISIN:") else s) or None

def _s3_is_isin(v: str) -> bool:
    return bool(re.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}[0-9]", str(v).strip().upper()))


def _s3_is_ric(v: str) -> bool:
    x = str(v).strip().upper()
    if not x:
        return False
    if _s3_is_isin(x):
        return False
    # Accept common RIC shapes (e.g., VOD.L, RICs with =, or plain mnemonic codes)
    return bool(re.search(r"[A-Z]", x)) and len(x) <= 32


def _s3_pick_first_valid(series: pd.Series, id_type: str) -> Optional[Tuple[str, str]]:
    want = str(id_type).upper().strip()
    for v in series:
        if pd.isna(v):
            continue
        vv = str(v).strip()
        if not vv:
            continue
        if want == "ISIN":
            vv = _s3_norm_isin(vv)
            if not vv:
                continue
            if not _s3_is_isin(vv):
                continue
        elif want == "RIC":
            if not _s3_is_ric(vv):
                continue
        return (want, vv)
    return None


def _s3_build_company_candidates_for_asof(df: pd.DataFrame, asof: pd.Timestamp) -> List[Tuple[str, str]]:
    """Build max-4 fallback ranks per company and as-of date.

    Rank order (requested):
    1) current RIC from exactly this quarter/date
    2) current ISIN from exactly this quarter/date
    3) newest available RIC for this firm_id
    4) newest available ISIN for this firm_id
    """
    q = df.copy()
    q["date"] = pd.to_datetime(q["date"], errors="coerce").dt.normalize()
    q = q.dropna(subset=["date"]).sort_values("date")
    asof_ts = pd.to_datetime(asof, errors="coerce").normalize()

    cur = q[q["date"] == asof_ts].copy()

    # Rank 1: current-quarter RIC (prefer RIC_current, then RIC)
    r1 = _s3_pick_first_valid(cur.get("RIC_current", pd.Series(dtype="object")), "RIC")
    if r1 is None:
        r1 = _s3_pick_first_valid(cur.get("RIC", pd.Series(dtype="object")), "RIC")

    # Rank 2: current-quarter ISIN
    r2 = _s3_pick_first_valid(cur.get("ISIN", pd.Series(dtype="object")), "ISIN")

    # Latest snapshot for firm-level fallback ranks 3/4
    latest = q.sort_values("date", ascending=False)

    # Rank 3: newest RIC (prefer RIC_current, then RIC)
    r3 = _s3_pick_first_valid(latest.get("RIC_current", pd.Series(dtype="object")), "RIC")
    if r3 is None:
        r3 = _s3_pick_first_valid(latest.get("RIC", pd.Series(dtype="object")), "RIC")

    # Rank 4: newest ISIN
    r4 = _s3_pick_first_valid(latest.get("ISIN", pd.Series(dtype="object")), "ISIN")

    out: List[Tuple[str, str]] = []
    seen = set()
    for item in (r1, r2, r3, r4):
        if item is None:
            continue
        key = (str(item[0]).upper(), str(item[1]).strip().upper())
        if key in seen:
            continue
        seen.add(key)
        out.append((str(item[0]).upper(), str(item[1]).strip()))

    return out


# ---- Per-company cache (extensible across index changes) ---
_s3_cache_mem: Dict[str, pd.DataFrame] = {}

def _s3_cache_file(firm_id: str) -> Path:
    key = str(firm_id).upper().strip()
    h = hashlib.sha1(key.encode()).hexdigest()[:16]
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", key[:60])
    return S3_CACHE_DIR / f"{clean}__{h}__{CACHE_VERSION_S3}.parquet"

def _s3_norm_cache(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["date"] = pd.to_datetime(x.get("date"), errors="coerce").dt.normalize()
    for c in TARGET_COLS_S3:
        if c not in x.columns:
            x[c] = np.nan
        x[c] = pd.to_numeric(x[c], errors="coerce")
    x = x.dropna(subset=["date"]).drop_duplicates("date", keep="last").sort_values("date")
    return x[["date"] + TARGET_COLS_S3].reset_index(drop=True)

def _s3_get_cache(firm_id: str) -> pd.DataFrame:
    if firm_id not in _s3_cache_mem:
        try:
            _s3_cache_mem[firm_id] = _s3_load_company_cache_with_legacy(firm_id)
        except Exception:
            _s3_cache_mem[firm_id] = pd.DataFrame(columns=["date"] + TARGET_COLS_S3)
    return _s3_cache_mem[firm_id]

def _s3_resolve_store_read_path() -> Optional[Path]:
    """Prefer current store, fallback to legacy backup store."""
    if S3_ROWS_PATH.exists():
        return S3_ROWS_PATH
    if S3_ROWS_PATH_LEGACY.exists():
        return S3_ROWS_PATH_LEGACY
    return None


def _s3_load_company_cache_with_legacy(firm_id: str) -> pd.DataFrame:
    """Load current per-company cache, fallback/merge with legacy cache variants."""
    key = str(firm_id).upper().strip()
    h = hashlib.sha1(key.encode()).hexdigest()[:16]
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", key[:60])

    frames = []

    # Current-version file first.
    p_cur = _s3_cache_file(firm_id)
    if p_cur.exists():
        try:
            frames.append(_s3_norm_cache(pd.read_parquet(p_cur)))
        except Exception:
            pass

    # Legacy variants by cache version suffix.
    for p in S3_CACHE_DIR.glob(f"{clean}__{h}__*.parquet"):
        if p == p_cur:
            continue
        try:
            frames.append(_s3_norm_cache(pd.read_parquet(p)))
        except Exception:
            continue

    # Last-resort legacy naming without version suffix.
    p_legacy_plain = S3_CACHE_DIR / f"{clean}__{h}.parquet"
    if p_legacy_plain.exists():
        try:
            frames.append(_s3_norm_cache(pd.read_parquet(p_legacy_plain)))
        except Exception:
            pass

    if not frames:
        return pd.DataFrame(columns=["date"] + TARGET_COLS_S3)

    merged = pd.concat(frames, ignore_index=True)
    return _s3_norm_cache(merged)


def _s3_load_no_data_quarters() -> set[str]:
    p = S3_NODATA_Q_PATH if S3_NODATA_Q_PATH.exists() else S3_NODATA_Q_PATH_LEGACY
    if not p.exists():
        return set()
    try:
        obj = json.loads(p.read_text())
        if isinstance(obj, dict):
            vals = obj.get("quarters", [])
        elif isinstance(obj, list):
            vals = obj
        else:
            vals = []
        out = set()
        for v in vals:
            try:
                out.add(pd.to_datetime(v).date().isoformat())
            except Exception:
                continue
        return out
    except Exception:
        return set()


def _s3_save_no_data_quarters(quarters: set[str]) -> None:
    payload = {
        "updated_at": pd.Timestamp.utcnow().isoformat(),
        "quarters": sorted(set(str(q) for q in quarters)),
    }
    tmp = S3_NODATA_Q_PATH.with_suffix(S3_NODATA_Q_PATH.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2))
    tmp.replace(S3_NODATA_Q_PATH)


def _s3_load_bad_ids() -> set[str]:
    p = S3_BAD_IDS_PATH if S3_BAD_IDS_PATH.exists() else S3_BAD_IDS_PATH_LEGACY
    if not p.exists():
        return set()
    try:
        obj = json.loads(p.read_text())
        if isinstance(obj, dict):
            vals = obj.get("bad_ids", [])
        elif isinstance(obj, list):
            vals = obj
        else:
            vals = []
        return set(str(v).strip().upper() for v in vals if str(v).strip())
    except Exception:
        return set()


def _s3_save_bad_ids(bad_ids: set[str]) -> None:
    payload = {
        "updated_at": pd.Timestamp.utcnow().isoformat(),
        "bad_ids": sorted(set(str(v).strip().upper() for v in bad_ids if str(v).strip())),
    }
    tmp = S3_BAD_IDS_PATH.with_suffix(S3_BAD_IDS_PATH.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2))
    tmp.replace(S3_BAD_IDS_PATH)


def _s3_put_cache(firm_id: str, df: pd.DataFrame) -> None:
    norm = _s3_norm_cache(df)
    _s3_cache_mem[firm_id] = norm
    p = _s3_cache_file(firm_id)
    tmp = p.with_suffix(p.suffix + ".tmp")
    norm.to_parquet(tmp, index=False)
    tmp.replace(p)

# ---- LSEG API call with rate-limit backoff -----------------
# ---- LSEG network guard --------------------------------------
def _s3_is_protocol_or_proxy_error(exc: Exception) -> bool:
    msg = f"{type(exc).__name__}: {exc}".lower()
    tokens = [
        "remoteprotocolerror",
        "illegal status line",
        "localhost:9000/api/udf",
        "check if desktop is running",
        "no proxy address identified",
        "connection refused",
    ]
    return any(t in msg for t in tokens)


def _s3_disable_network(reason: str) -> None:
    global _s3_disable_network_runtime, _s3_disable_network_reason, _s3_disable_network_warned
    msg = str(reason).replace("\n", " ").strip()
    if len(msg) > 180:
        msg = msg[:177] + "..."
    _s3_disable_network_runtime = True
    _s3_disable_network_reason = msg
    if not _s3_disable_network_warned:
        print(f"[WARN S3] Disabling network pulls for this run; continuing from cache/store only. reason={msg}", flush=True)
        _s3_disable_network_warned = True

def _s3_call_get_data(universe: List[str], date, raise_on_error: bool = False) -> pd.DataFrame:
    """ld.get_data for EPS snapshot at date; retries on 429 with backoff."""
    global _s3_disable_network_runtime
    if _s3_disable_network_runtime:
        return pd.DataFrame()

    asof_str = pd.to_datetime(date).strftime("%Y-%m-%d")
    cooldown = RATE_LIMIT_COOLDOWN_SEC_S3
    last_err = None
    for attempt in range(MAX_RETRIES_S3 + 1):
        try:
            ex = _cf.ThreadPoolExecutor(max_workers=1)
            try:
                fut = ex.submit(
                    ld.get_data,
                    universe=list(universe),
                    fields=list(EPS_FIELDS_MAP.values()),
                    parameters={"SDate": asof_str, "EDate": asof_str},
                )
                raw = fut.result(timeout=float(S3_CALL_TIMEOUT_SEC))
            finally:
                # Do not block on hung worker thread when timeout is hit.
                ex.shutdown(wait=False, cancel_futures=True)
            return pd.DataFrame(raw) if raw is not None else pd.DataFrame()
        except Exception as e:
            last_err = e
            if AUTO_DISABLE_NETWORK_ON_PROTOCOL_ERR_S3 and _s3_is_protocol_or_proxy_error(e):
                _s3_disable_network(f"{type(e).__name__}: {e}")
                return pd.DataFrame()
            if isinstance(e, TimeoutError):
                print(f"[WARN S3 timeout] attempt={attempt+1}/{MAX_RETRIES_S3+1} sec={S3_CALL_TIMEOUT_SEC:.0f} ids={len(universe)}", flush=True)
            msg = str(e).lower()
            is_rl = "too many requests" in msg or bool(re.search(r"\b429\b", msg))
            if is_rl:
                if FAIL_FAST_ON_RATE_LIMIT_S3:
                    raise
                pause = min(cooldown, RATE_LIMIT_HARD_PAUSE_SEC_S3)
                print(f"[WARN S3 rate-limit] attempt={attempt+1}/{MAX_RETRIES_S3+1} sleep={pause:.0f}s | {e}", flush=True)
                time.sleep(pause)
                cooldown *= RATE_LIMIT_MULTIPLIER_S3
                continue
            if attempt >= MAX_RETRIES_S3:
                break
            time.sleep(1.0 * (2 ** attempt))
    if last_err is not None:
        # In recursive isolation mode (raise_on_error=True), failures are expected
        # and handled by caller via split strategy; avoid warning spam.
        if not raise_on_error:
            print(f"[WARN S3 get_data failed] {type(last_err).__name__}: {last_err}", flush=True)
        if raise_on_error:
            raise last_err
    return pd.DataFrame()

# ---- Response parser ---------------------------------------
def _s3_num_scalar(v, prefer_idx: int | None = None):
    """Normalize arbitrary LSEG cell payloads to a numeric scalar.

    If payload is sequence-like, prefer_idx can select the FY-position instead of
    always taking the first element.
    """
    if isinstance(v, pd.DataFrame):
        v = v.stack(dropna=True)

    if isinstance(v, pd.Series):
        s = v.dropna()
        if s.empty:
            return np.nan
        if prefer_idx is not None and 0 <= int(prefer_idx) < len(s):
            v = s.iloc[int(prefer_idx)]
        else:
            v = s.iloc[0]
    elif isinstance(v, (list, tuple, np.ndarray)):
        seq = pd.Series(list(v)).dropna()
        if seq.empty:
            return np.nan
        if prefer_idx is not None and 0 <= int(prefer_idx) < len(seq):
            v = seq.iloc[int(prefer_idx)]
        else:
            v = seq.iloc[0]

    return pd.to_numeric(v, errors="coerce")

def _s3_parse_response(raw_df: pd.DataFrame, ids: List[str]) -> Dict[str, Dict]:
    """Parse LSEG response -> {pull_id_upper: {EPS_FY1: float, ...}}.
    Columns assigned by position (same order as EPS_FIELDS_MAP).
    Instrument matching: exact -> contains -> single-id fallback."""
    if raw_df is None or raw_df.empty:
        return {}
    x = raw_df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [" | ".join(str(v) for v in tup if v).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]

    # Identify instrument column
    inst_col = x.columns[0]
    for c in x.columns:
        if c.lower() in {"instrument", "ric", "isin"} or "instrument" in c.lower():
            inst_col = c
            break

    inst_pos    = list(x.columns).index(inst_col)
    value_pos   = [j for j in range(len(x.columns)) if j != inst_pos]
    ids_norm    = {str(i).strip().upper(): str(i).strip() for i in ids}
    inst_series = x[inst_col].astype("string").str.strip().str.upper()
    out: Dict[str, Dict] = {}

    for row_idx in range(len(x)):
        inst = inst_series.iloc[row_idx]
        if pd.isna(inst):
            continue
        # 1) Exact match
        matched = ids_norm.get(inst)
        # 2) Contains fallback (LSEG sometimes strips/decorates identifiers)
        if matched is None:
            for norm_id in ids_norm:
                if re.search(re.escape(norm_id), inst, re.IGNORECASE) or \
                   re.search(re.escape(inst), norm_id, re.IGNORECASE):
                    matched = ids_norm[norm_id]
                    break
        # 3) Single-id fallback
        if matched is None and len(ids) == 1:
            matched = list(ids_norm.values())[0]
        if matched is None:
            continue

        values = {}
        for i, c in enumerate(TARGET_COLS_S3):
            if i < len(value_pos):
                # Positional access avoids duplicate-column-label collisions in LSEG responses.
                raw_cell = x.iloc[row_idx, value_pos[i]]
                values[c] = _s3_num_scalar(raw_cell, prefer_idx=i)
            else:
                values[c] = np.nan
        out[matched.strip().upper()] = values

    return out

_s3_last_fetch_error: str = ""

def _s3_fetch_parsed_recursive(ids: List[str], date, max_depth: int = 10) -> Tuple[Dict[str, Dict], List[str]]:
    """Fetch EPS robustly: split failing groups to isolate bad identifiers."""
    ids = [str(i).strip() for i in ids if str(i).strip()]
    if not ids:
        return {}, []

    global _s3_last_fetch_error
    try:
        raw = _s3_call_get_data(ids, date, raise_on_error=True)
    except Exception as e:
        _s3_last_fetch_error = f"{type(e).__name__}: {e}"
        if S3_MINIMIZE_REQUESTS:
            return {}, ids
        if len(ids) == 1 or max_depth <= 0:
            return {}, ids[:1]
        mid = max(1, len(ids) // 2)
        p_l, b_l = _s3_fetch_parsed_recursive(ids[:mid], date, max_depth=max_depth - 1)
        p_r, b_r = _s3_fetch_parsed_recursive(ids[mid:], date, max_depth=max_depth - 1)
        out = {}
        out.update(p_l)
        out.update(p_r)
        return out, (b_l + b_r)

    if raw is None or raw.empty:
        return {}, []

    parsed = _s3_parse_response(raw, ids)
    if parsed:
        return parsed, []

    if S3_MINIMIZE_REQUESTS:
        return {}, []
    if len(ids) == 1 or max_depth <= 0:
        return {}, []
    mid = max(1, len(ids) // 2)
    p_l, b_l = _s3_fetch_parsed_recursive(ids[:mid], date, max_depth=max_depth - 1)
    p_r, b_r = _s3_fetch_parsed_recursive(ids[mid:], date, max_depth=max_depth - 1)
    out = {}
    out.update(p_l)
    out.update(p_r)
    return out, (b_l + b_r)


# ---- Store management --------------------------------------
_S3_PANEL_COLS = ["firm_id", "date"] + TARGET_COLS_S3

def _s3_prep_store(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=_S3_PANEL_COLS)
    x = df.copy()
    for c in _S3_PANEL_COLS:
        if c not in x.columns:
            x[c] = np.nan if c in TARGET_COLS_S3 else pd.NA
    x["date"] = pd.to_datetime(x["date"], errors="coerce").dt.normalize()
    for c in TARGET_COLS_S3:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return (x[_S3_PANEL_COLS]
            .dropna(subset=["firm_id", "date"])
            .drop_duplicates(["firm_id", "date"], keep="last")
            .reset_index(drop=True))

def _s3_upsert_store(store: pd.DataFrame, new_rows: pd.DataFrame) -> pd.DataFrame:
    s, n = _s3_prep_store(store), _s3_prep_store(new_rows)
    if n.empty:
        return s
    if s.empty:
        return n
    keep = s.merge(n[["firm_id", "date"]], on=["firm_id", "date"],
                   how="left", indicator=True)
    keep = keep[keep["_merge"] == "left_only"].drop(columns=["_merge"])
    return _s3_prep_store(pd.concat([keep, n], ignore_index=True))

# ---- Build request table -----------------------------------
euro500_eps = euro500.copy()
euro500_eps["date"] = _s3_asof(euro500_eps)
for c in ("ISIN", "RIC_current", "RIC", "pull_id", "id_type", "firm_id"):
    if c in euro500_eps.columns:
        euro500_eps[c] = _s3_clean(euro500_eps[c])

if "firm_id" not in euro500_eps.columns or not euro500_eps["firm_id"].notna().any():
    _fid = _s3_clean(euro500_eps.get("ISIN",        pd.Series(pd.NA, index=euro500_eps.index, dtype="string")))
    _fid = _fid.fillna(_s3_clean(euro500_eps.get("RIC_current", pd.Series(pd.NA, index=euro500_eps.index, dtype="string"))))
    _fid = _fid.fillna(_s3_clean(euro500_eps.get("RIC",         pd.Series(pd.NA, index=euro500_eps.index, dtype="string"))))
    euro500_eps["firm_id"] = "CID:" + _fid.astype("string")
else:
    euro500_eps["firm_id"] = _s3_clean(euro500_eps["firm_id"])

_req_cols = [c for c in ["firm_id", "date", "ISIN", "RIC_current", "RIC", "pull_id", "id_type"]
             if c in euro500_eps.columns]
req_all = (euro500_eps[_req_cols]
           .dropna(subset=["firm_id", "date"])
           .drop_duplicates()
           .reset_index(drop=True))

req_all = req_all[req_all["date"] >= S3_MIN_ASOF_DATE].copy().reset_index(drop=True)

if S3_APPEND_TODAY_ASOF and not req_all.empty:
    today_asof = pd.Timestamp.today().normalize()
    latest_per_firm = (
        req_all.sort_values(["firm_id", "date"])
        .groupby("firm_id", as_index=False)
        .tail(1)
        .copy()
    )
    latest_per_firm["date"] = today_asof
    req_all = (
        pd.concat([req_all, latest_per_firm], ignore_index=True)
        .drop_duplicates()
        .sort_values(["firm_id", "date"])
        .reset_index(drop=True)
    )
    print(f"[S3] Appended today's as-of date for validation: {today_asof.date()} (+{len(latest_per_firm)} firm rows)")

if req_all.empty:
    raise ValueError(f"No valid (firm_id, date) rows found for Step 3 after filter >= {S3_MIN_ASOF_DATE.date()}.")

# Flat candidate table  (firm_id x date x rank), max 4 ranks per requested as-of.
_cand_rows = []
for fid, g in req_all.groupby("firm_id", sort=False):
    g = g.copy()
    dates = g["date"].dropna().dt.normalize().unique()
    for asof in dates:
        cands = _s3_build_company_candidates_for_asof(g, pd.to_datetime(asof).normalize())
        for rank, (id_type, pull_id) in enumerate(cands, start=1):
            _cand_rows.append({"firm_id": str(fid),
                                "date": pd.to_datetime(asof).normalize(),
                                "rank": rank, "id_type": id_type, "pull_id": str(pull_id)})

cand_df  = (pd.DataFrame(_cand_rows) if _cand_rows
            else pd.DataFrame(columns=["firm_id", "date", "rank", "id_type", "pull_id"]))
max_rank = int(cand_df["rank"].max()) if not cand_df.empty else 0

print(f"Step 3 -- firm x asof rows: {len(req_all)}")
print("S3 resume mode: OFF (full asof range scan)") if not RESUME_FROM_LAST_FILLED_S3 else print("S3 resume mode: ON")
print(f"S3 force refresh rows: {FORCE_REFRESH_S3} (store prefill remains enabled)")
print(f"Firms: {req_all['firm_id'].nunique()}  |  "
      f"asof range: {req_all['date'].min().date()} -> {req_all['date'].max().date()}")
print(f"Max ID-fallback ranks: {max_rank}  |  "
      f"Mode: {'CACHE_ONLY (LSEG disabled)' if DISABLE_LSEG_PULL_S3 else ('CACHE_ONLY' if CACHE_ONLY_S3 else 'CACHE+NETWORK')}")
print("Rank order per quarter: 1=current RIC, 2=current ISIN, 3=newest RIC, 4=newest ISIN")
if cand_df.empty:
    raise ValueError("No candidate identifiers.")

# ---- Load existing store & build working panel -------------
s3_store = pd.DataFrame(columns=_S3_PANEL_COLS)
_s3_store_read_path = _s3_resolve_store_read_path()
if _s3_store_read_path is not None:
    try:
        s3_store = _s3_prep_store(pd.read_parquet(_s3_store_read_path))
        print(f"Loaded S3 store from {_s3_store_read_path.name}: {len(s3_store)} rows ({s3_store['firm_id'].nunique()} firms)")
    except Exception as e:
        print(f"[WARN] Could not load S3 store from {_s3_store_read_path}: {e}")

panel = req_all[["firm_id", "date"]].drop_duplicates().copy()
for c in TARGET_COLS_S3:
    panel[c] = np.nan

if not s3_store.empty:
    panel = panel.merge(s3_store[["firm_id", "date"] + TARGET_COLS_S3],
                        on=["firm_id", "date"], how="left", suffixes=("", "_old"))
    for c in TARGET_COLS_S3:
        panel[c] = (pd.to_numeric(panel[f"{c}_old"], errors="coerce")
                    .combine_first(pd.to_numeric(panel[c], errors="coerce")))
    panel = panel.drop(columns=[c for c in panel.columns if c.endswith("_old")])

pre_filled_partial = int(panel[REQUIRED_COLS_S3].notna().any(axis=1).sum())
pre_filled_full = int(panel[REQUIRED_COLS_S3].notna().all(axis=1).sum())
print(f"Pre-filled from store: partial={pre_filled_partial}/{len(panel)} | full={pre_filled_full}/{len(panel)}")

s3_no_data_quarters = _s3_load_no_data_quarters() if S3_USE_NODATA_QUARTER_LOG else set()
s3_bad_ids = _s3_load_bad_ids()
if S3_USE_NODATA_QUARTER_LOG:
    print(f"Known no-data quarters: {len(s3_no_data_quarters)}")
print(f"Known bad IDs: {len(s3_bad_ids)}")

# ---- Main pull loop ----------------------------------------
run_t0        = time.time()
progress_rows = []

if not CACHE_ONLY_S3:
    if S3_DP_PROXY_BASE_URL:
        os.environ["DP_PROXY_BASE_URL"] = str(S3_DP_PROXY_BASE_URL).strip()
        print(f"[S3] DP_PROXY_BASE_URL={os.environ['DP_PROXY_BASE_URL']}", flush=True)
    try:
        ld.open_session()
    except Exception as e:
        if AUTO_DISABLE_NETWORK_ON_PROTOCOL_ERR_S3 and _s3_is_protocol_or_proxy_error(e):
            _s3_disable_network(f"open_session {type(e).__name__}: {e}")
        else:
            raise

if S3_FAIL_FAST_IF_NO_NETWORK_AND_NO_CACHE and (CACHE_ONLY_S3 or _s3_disable_network_runtime) and pre_filled_partial == 0:
    raise RuntimeError(
        "Step 3 cannot proceed: no network available and no cached EPS rows in store. "
        "Fix LSEG Workspace/proxy (localhost:9000) or run with a populated Step-3 cache/backup."
    )

try:
    dates_all = sorted(panel["date"].dropna().unique().tolist())
    dates = dates_all

    if RESUME_FROM_LAST_FILLED_S3:
        per_row_any = panel[REQUIRED_COLS_S3].notna().any(axis=1)
        per_date_any = per_row_any.groupby(panel["date"]).any()
        filled_dates = sorted(pd.to_datetime(per_date_any[per_date_any].index, errors="coerce"))
        if filled_dates:
            last_filled = pd.to_datetime(filled_dates[-1]).normalize()
            if INCLUDE_LAST_FILLED_S3:
                dates = [d for d in dates_all if pd.to_datetime(d).normalize() >= last_filled]
                print(f"[S3 resume] last quarter with EPS values: {last_filled.date()} -> include this date", flush=True)
            else:
                dates = [d for d in dates_all if pd.to_datetime(d).normalize() > last_filled]
                print(f"[S3 resume] last quarter with EPS values: {last_filled.date()} -> start from next date", flush=True)

    if not dates:
        print("[S3] No pending asof dates after resume filter.", flush=True)

    n_date_batches = int(np.ceil(len(dates) / S3_DATE_BATCH_SIZE)) if dates else 0

    for d_ix, asof in enumerate(dates, 1):
        asof_ts   = pd.to_datetime(asof).normalize()
        date_mask = panel["date"] == asof_ts
        n_total   = int(date_mask.sum())
        b_ix = int(np.ceil(d_ix / S3_DATE_BATCH_SIZE)) if S3_DATE_BATCH_SIZE > 0 else 1
        b_start = ((b_ix - 1) * S3_DATE_BATCH_SIZE) + 1 if S3_DATE_BATCH_SIZE > 0 else d_ix
        b_end = min(len(dates), b_ix * S3_DATE_BATCH_SIZE) if S3_DATE_BATCH_SIZE > 0 else d_ix
        if d_ix == b_start:
            print(f"\n[BATCH {b_ix}/{n_date_batches}] quarters={b_end - b_start + 1} idx={b_start}-{b_end}", flush=True)
        q_label  = f"{asof_ts.year}Q{asof_ts.quarter}"
        print(f"  [QUARTER {d_ix}/{len(dates)}] {q_label} ({asof_ts.date()}) rows={n_total}", flush=True)

        quarter_key = asof_ts.date().isoformat()
        quarter_prefilled_any = bool(panel.loc[date_mask, REQUIRED_COLS_S3].notna().any(axis=1).any())
        skip_quarter_network = (
            S3_USE_NODATA_QUARTER_LOG
            and quarter_key in s3_no_data_quarters
            and (not FORCE_REFRESH_S3)
            and (not quarter_prefilled_any)
        )
        if skip_quarter_network:
            print(f"    known no-data quarter log hit -> skip network pulls for {quarter_key}", flush=True)

        quarter_network_attempted = False
        quarter_network_parsed_hits = 0

        for rank in range(1, max_rank + 1):
            if skip_quarter_network:
                if rank == 1 and max_rank > 1:
                    print(f"    cached no-data path: stop after r1; skip r2-r{max_rank}", flush=True)
                break
            unres_mask = date_mask & panel[REQUIRED_COLS_S3].isna().any(axis=1)
            n_unres    = int(unres_mask.sum())
            if n_unres == 0:
                print(f"    r{rank}: miss=0 (complete)", flush=True)
                break

            # Candidates for this rank / date / unresolved firms
            unres_firms = set(panel.loc[unres_mask, "firm_id"].astype(str))
            rank_cands  = cand_df[(cand_df["date"] == asof_ts) &
                                  (cand_df["rank"] == rank) &
                                  (cand_df["firm_id"].astype(str).isin(unres_firms))].copy()
            if rank_cands.empty:
                continue

            # pid -> [firm_ids] map (handles multiple firms sharing same ISIN/RIC)
            pid_to_firms: Dict[str, List[str]] = {}
            for _, row in rank_cands.iterrows():
                pid_key = str(row["pull_id"]).strip().upper()
                pid_to_firms.setdefault(pid_key, []).append(str(row["firm_id"]))

            # Check per-company caches first (exact date only)
            pids_need_network: List[str] = []
            cache_hit_count = 0
            for _, row in rank_cands.iterrows():
                fid = str(row["firm_id"])
                pid = str(row["pull_id"]).strip().upper()
                cached = _s3_get_cache(fid)
                if not cached.empty:
                    hit = cached[cached["date"] == asof_ts]
                    if not hit.empty and hit[REQUIRED_COLS_S3].notna().values.any():
                        m = (panel["firm_id"] == fid) & (panel["date"] == asof_ts)
                        for c in TARGET_COLS_S3:
                            v = _s3_num_scalar(hit[c].iloc[-1])
                            if pd.notna(v):
                                panel.loc[m & panel[c].isna(), c] = v
                        cache_hit_count += 1
                        continue
                pids_need_network.append(pid)

            pids_need_network = sorted(set(pids_need_network))
            skipped_bad_ids = 0
            if s3_bad_ids and (not FORCE_REFRESH_S3):
                before_bad_filter = len(pids_need_network)
                pids_need_network = [pid for pid in pids_need_network if pid not in s3_bad_ids]
                skipped_bad_ids = before_bad_filter - len(pids_need_network)
                if skipped_bad_ids and not S3_COMPACT_LOG:
                    print(f"    bad-id cache skip: {skipped_bad_ids} ids", flush=True)
            if not S3_COMPACT_LOG:
                print(f"  rank={rank}: unresolved={n_unres} "
                      f"cache_hits={cache_hit_count} network_pids={len(pids_need_network)}", flush=True)

            # Network pull
            if pids_need_network and not CACHE_ONLY_S3 and (not _s3_disable_network_runtime):
                quarter_network_attempted = True
                chunks = [pids_need_network[i:i + BATCH_SIZE_S3]
                          for i in range(0, len(pids_need_network), BATCH_SIZE_S3)]
                sum_chunk_sec = 0.0
                for ch_ix, chunk in enumerate(chunks, 1):
                    t0 = time.time()
                    parsed, bad_ids = _s3_fetch_parsed_recursive(chunk, asof_ts, max_depth=S3_SPLIT_MAX_DEPTH_S3)
                    dt = time.time() - t0
                    sum_chunk_sec += dt
                    if not S3_COMPACT_LOG:
                        print(f"    chunk {ch_ix}/{len(chunks)} ids={len(chunk)} got={len(parsed)} bad={len(bad_ids)} sec={dt:.1f}", flush=True)

                    if bad_ids and (not FORCE_REFRESH_S3):
                        new_bad = {str(x).strip().upper() for x in bad_ids if str(x).strip()}
                        add_bad = len(new_bad - s3_bad_ids)
                        if add_bad:
                            s3_bad_ids.update(new_bad)
                            _s3_save_bad_ids(s3_bad_ids)
                    for pid_upper, values in parsed.items():
                        quarter_network_parsed_hits += 1
                        for fid in pid_to_firms.get(pid_upper, []):
                            # Update working panel (fill NaNs only)
                            m = (panel["firm_id"] == fid) & (panel["date"] == asof_ts)
                            for c in TARGET_COLS_S3:
                                v = _s3_num_scalar(values.get(c))
                                if pd.notna(v):
                                    panel.loc[m & panel[c].isna(), c] = v
                            # Persist to per-company cache
                            existing = _s3_get_cache(fid)
                            new_row  = pd.DataFrame([{"date": asof_ts,
                                                      **{c: values.get(c, np.nan) for c in TARGET_COLS_S3}}])
                            if existing.empty:
                                _s3_put_cache(fid, new_row)
                            else:
                                _s3_put_cache(fid, pd.concat([existing, new_row], ignore_index=True))

                    if bad_ids and DEBUG_VERBOSE_S3:
                        show = ', '.join(bad_ids[:5])
                        more = '' if len(bad_ids) <= 5 else f' (+{len(bad_ids)-5} more)'
                        print(f"      unresolved identifiers in chunk: [{show}]{more}", flush=True)
                        if len(parsed) == 0 and len(bad_ids) == len(chunk):
                            err_hint = _s3_last_fetch_error if str(_s3_last_fetch_error).strip() else "unknown"
                            print(f"      chunk failure reason (last): {err_hint}", flush=True)

            elif pids_need_network and (CACHE_ONLY_S3 or _s3_disable_network_runtime):
                if not S3_COMPACT_LOG:
                    if _s3_disable_network_runtime:
                        print(f"    network disabled at runtime, skipping {len(pids_need_network)} ids | reason={_s3_disable_network_reason}", flush=True)
                    else:
                        print(f"    CACHE_ONLY_S3=True, skipping {len(pids_need_network)} network ids", flush=True)

            if S3_COMPACT_LOG:
                if pids_need_network and not CACHE_ONLY_S3 and (not _s3_disable_network_runtime):
                    bad_part = f" bad_skip={skipped_bad_ids}" if skipped_bad_ids else ""
                    print(
                        f"    r{rank}: miss={n_unres} cache={cache_hit_count} net={len(pids_need_network)} "
                        f"chunks={len(chunks)} t={sum_chunk_sec:.1f}s{bad_part}",
                        flush=True,
                    )
                elif pids_need_network and (CACHE_ONLY_S3 or _s3_disable_network_runtime):
                    reason = _s3_disable_network_reason if _s3_disable_network_runtime else "CACHE_ONLY_S3=True"
                    bad_part = f" bad_skip={skipped_bad_ids}" if skipped_bad_ids else ""
                    print(
                        f"    r{rank}: miss={n_unres} cache={cache_hit_count} net_skipped={len(pids_need_network)} reason={reason}{bad_part}",
                        flush=True,
                    )
                else:
                    bad_part = f" bad_skip={skipped_bad_ids}" if skipped_bad_ids else ""
                    print(
                        f"    r{rank}: miss={n_unres} cache={cache_hit_count} net=0{bad_part}",
                        flush=True,
                    )

            # In cache-only / runtime network-disabled mode, additional ranks do not add information:
            # cache lookup is firm+date-based and independent of identifier rank.
            if S3_STOP_AFTER_R1_WHEN_NO_NETWORK and rank == 1 and (CACHE_ONLY_S3 or _s3_disable_network_runtime):
                if max_rank > 1:
                    print(f"    cache-only path: stop after r1; skip r2-r{max_rank}", flush=True)
                # Fail fast when the run has no usable cache either.
                if S3_FAIL_FAST_IF_NO_NETWORK_AND_NO_CACHE and _s3_disable_network_runtime and cache_hit_count == 0 and n_unres == n_total:
                    raise RuntimeError(
                        "Step 3 aborted early: network disabled by LSEG proxy/protocol error and no cache hits "
                        f"for {q_label}. Cannot fill EPS without network or backup cache."
                    )
                break

        # ---- Progress for this date ----------------------------
        found    = {c: int(panel.loc[date_mask, c].notna().sum()) for c in TARGET_COLS_S3}
        partial  = int(panel.loc[date_mask, REQUIRED_COLS_S3].notna().any(axis=1).sum())
        full     = int(panel.loc[date_mask, REQUIRED_COLS_S3].notna().all(axis=1).sum())
        elapsed  = time.time() - run_t0
        # q_label already defined in quarter header
        found_str = "  ".join(f"{c}={found[c]}/{n_total}" for c in TARGET_COLS_S3)
        partial_pct = (100.0 * partial / n_total) if n_total else np.nan
        full_pct = (100.0 * full / n_total) if n_total else np.nan
        if S3_COMPACT_OUTPUT:
            print(f"    result: partial={partial_pct:.1f}% full={full_pct:.1f}% elapsed={elapsed/60:.1f}m", flush=True)
        else:
            print(f"[S3] {q_label} ({asof_ts.date()}) partial={partial}/{n_total} ({partial_pct:.1f}%) | "
                  f"full={full}/{n_total} ({full_pct:.1f}%) | {found_str} | elapsed={elapsed/60:.1f}m", flush=True)

        # Persist quarter-level no-data marker to avoid repeated empty pulls on later runs.
        if (
            S3_USE_NODATA_QUARTER_LOG
            and quarter_network_attempted
            and (quarter_network_parsed_hits == 0)
            and (partial == 0)
            and (not FORCE_REFRESH_S3)
        ):
            if quarter_key not in s3_no_data_quarters:
                s3_no_data_quarters.add(quarter_key)
                _s3_save_no_data_quarters(s3_no_data_quarters)
                print(f"    no-data quarter logged: {quarter_key}", flush=True)

        progress_rows.append({"quarter": q_label,
                               "date": asof_ts.date().isoformat(),
                               "rows": n_total,
                               "partial": partial,
                               "full": full,
                               "partial_pct": round(float(partial_pct), 2) if pd.notna(partial_pct) else np.nan,
                               "full_pct": round(float(full_pct), 2) if pd.notna(full_pct) else np.nan,
                               "elapsed_min": round(elapsed / 60, 1),
                               **{f"found_{c}": found[c] for c in TARGET_COLS_S3}})

        # ---- Incremental save ----------------------------------
        s3_store = _s3_upsert_store(s3_store, panel.loc[date_mask].copy())
        _tmp = S3_ROWS_PATH.with_suffix(S3_ROWS_PATH.suffix + ".tmp")
        s3_store.to_parquet(_tmp, index=False)
        _tmp.replace(S3_ROWS_PATH)
        S3_CKPT_PATH.write_text(json.dumps({
            "updated_at": pd.Timestamp.utcnow().isoformat(),
            "last_date":  asof_ts.date().isoformat(),
            "store_rows": len(s3_store),
        }, indent=2))

finally:
    pass  # session left open intentionally; ld.close_session() can trigger
          # pandas-truthiness issues in some LSEG internals on pandas 2.x / Python 3.13

# ---- Merge results & save output ---------------------------
out = _s3_prep_store(panel)

# Deterministic write: do not reuse legacy EPS values from older runs.
# This prevents stale/misaligned history from leaking into the final table.
euro500_eps = euro500_eps.drop(columns=[c for c in TARGET_COLS_S3 if c in euro500_eps.columns], errors="ignore")

euro500_eps = euro500_eps.merge(
    out[["firm_id", "date"] + TARGET_COLS_S3],
    on=["firm_id", "date"],
    how="left",
    validate="m:1",
)

for c in TARGET_COLS_S3:
    euro500_eps[c] = pd.to_numeric(euro500_eps[c], errors="coerce")

euro500_eps.to_parquet(EURO500_EPS_PATH, index=False)

# ---- Summary -----------------------------------------------
if progress_rows:
    display(pd.DataFrame(progress_rows))
if S3_SHOW_FILE_LOGS:
    print(f"\nSaved: {EURO500_EPS_PATH}  rows={len(euro500_eps)}")
# Step 3 parser sanity: EPS horizons should not be identically equal for almost all rows
_eps_cols_chk = [f"EPS_{h}" for h in HORIZONS if f"EPS_{h}" in euro500_eps.columns]
if len(_eps_cols_chk) >= 2:
    _eps_num = euro500_eps[_eps_cols_chk].apply(pd.to_numeric, errors='coerce')
    _same_all = _eps_num.notna().all(axis=1) & (_eps_num.nunique(axis=1) == 1)
    _all_nonnull = int(_eps_num.notna().all(axis=1).sum())
    _same_cnt = int(_same_all.sum())
    print(f"Step 3 parser sanity: rows(all FY equal among non-null rows) = {_same_cnt} / {_all_nonnull}")
    if _all_nonnull > 0 and (_same_cnt / _all_nonnull) > 0.95:
        print("WARNING: EPS FY horizons look suspiciously identical (>95% equal across FY1-5). Check raw mapping.")

if S3_SHOW_FILE_LOGS:
    print(f"S3 store: {S3_ROWS_PATH}  rows={len(s3_store)}")
    print(f"Cache dir: {S3_CACHE_DIR}")
    if S3_USE_NODATA_QUARTER_LOG:
        print(f"No-data quarter log: {S3_NODATA_Q_PATH}  entries={len(s3_no_data_quarters)}")
    print(f"Bad-ID log: {S3_BAD_IDS_PATH}  entries={len(s3_bad_ids)}")
for h in HORIZONS:
    c = f"EPS_{h}"
    if c in euro500_eps.columns:
        filled = int(euro500_eps[c].notna().sum())
        total  = len(euro500_eps)


In [ ]:
# Todays Forecasts for validation
df_wrangle = pd.read_parquet(
    Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/cache/cash_eps_act_step3_rows_v4.parquet")
)

### 3B. Coverage-Analyse EPS (`EPS_FY1`-`EPS_FY5`)

In [ ]:
from plot_style import (
    COLORS,
    set_global_plot_style,
    style_axes,
    style_legend,
    style_time_axis,
)

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

eps_cov = pd.read_parquet(EURO500_EPS_PATH).copy()

if 'date' not in eps_cov.columns:
    raise ValueError('Missing required date column for EPS coverage analysis.')
eps_cov['date'] = pd.to_datetime(eps_cov['date'], errors='coerce').dt.normalize()

eps_cov = eps_cov[eps_cov['date'].notna()].copy()
if eps_cov.empty:
    raise ValueError('No valid date values found for EPS coverage analysis.')

eps_cov['quarter'] = (eps_cov['date'] + pd.Timedelta(days=1)).dt.to_period('Q')
eps_cols = [c for c in ['EPS_FY1', 'EPS_FY2', 'EPS_FY3', 'EPS_FY4', 'EPS_FY5']]
act_col = 'EPS_CashAct'

for c in eps_cols + [act_col]:
    if c not in eps_cov.columns:
        eps_cov[c] = np.nan
    eps_cov[c] = pd.to_numeric(eps_cov[c], errors='coerce')

eps_cov['cash_eps_any'] = eps_cov[eps_cols].notna().any(axis=1)
eps_cov['cash_eps_all'] = eps_cov[eps_cols].notna().all(axis=1)
eps_cov['cash_eps_partial'] = eps_cov['cash_eps_any'] & (~eps_cov['cash_eps_all'])
eps_cov['cash_eps_non_null_n'] = eps_cov[eps_cols].notna().sum(axis=1)

agg_map = {
    'rows': ('quarter', 'size'),
    'cash_cash_eps_any_coverage': ('cash_eps_any', 'mean'),
    'cash_cash_eps_all_coverage': ('cash_eps_all', 'mean'),
    'cash_cash_eps_partial_coverage': ('cash_eps_partial', 'mean'),
}
for c in eps_cols:
    agg_map[f'{c}_coverage'] = (c, lambda s: s.notna().mean())
agg_map[f'{act_col}_coverage'] = (act_col, lambda s: s.notna().mean())

quarterly_eps_cov = (
    eps_cov.groupby('quarter', as_index=False)
    .agg(**agg_map)
    .sort_values('quarter')
)

cov_cols = [c for c in quarterly_eps_cov.columns if c.endswith('_coverage')]
quarterly_eps_cov[cov_cols] = (quarterly_eps_cov[cov_cols] * 100.0).round(2)
quarterly_eps_cov['quarter'] = quarterly_eps_cov['quarter'].astype(str)

focus_eps_cov = quarterly_eps_cov.tail(8).copy()

summary_eps = pd.DataFrame([
    {
        'rows_total': int(len(eps_cov)),
        'asof_min': pd.to_datetime(eps_cov['date']).min(),
        'asof_max': pd.to_datetime(eps_cov['date']).max(),
        'unique_pull_ric': int(eps_cov['pull_ric'].nunique(dropna=True)) if 'pull_ric' in eps_cov.columns else np.nan,
        'cash_cash_cash_eps_any_coverage_pct': round(float(eps_cov['cash_eps_any'].mean() * 100.0), 2),
        'cash_cash_cash_eps_all_coverage_pct': round(float(eps_cov['cash_eps_all'].mean() * 100.0), 2),
        'cash_cash_cash_eps_partial_coverage_pct': round(float(eps_cov['cash_eps_partial'].mean() * 100.0), 2),
    }
])

print('Cash EPS coverage summary:')
display(summary_eps)
print('Cash EPS quarterly coverage (%):')
display(quarterly_eps_cov)


# Plot: quarterly EPS coverage per horizon (FY1-FY5)
set_global_plot_style()
fy_cov_cols = [f'{c}_coverage' for c in eps_cols if f'{c}_coverage' in quarterly_eps_cov.columns]
act_cov_col = f'{act_col}_coverage'
plot_cols = fy_cov_cols + ([act_cov_col] if act_cov_col in quarterly_eps_cov.columns else [])
if plot_cols:
    fig, ax = plt.subplots(figsize=(12, 5.2))

    q_period = pd.PeriodIndex(quarterly_eps_cov['quarter'].astype(str), freq='Q')
    q_dates = q_period.to_timestamp(how='end')

    line_style = {
        'EPS_FY1_coverage': dict(color=COLORS['blue'], marker='o', markersize=3.0, linewidth=1.2, label='Cash EPS FY1'),
        'EPS_FY2_coverage': dict(color=COLORS['primary'], marker='o', markersize=3.0, linewidth=1.2, label='Cash EPS FY2'),
        'EPS_FY3_coverage': dict(color=COLORS['orange'], marker='o', markersize=3.0, linewidth=1.2, label='Cash EPS FY3'),
        'EPS_FY4_coverage': dict(color=COLORS['purple'], marker='o', markersize=3.0, linewidth=1.2, label='Cash EPS FY4'),
        'EPS_FY5_coverage': dict(color=COLORS['accent'], marker='o', markersize=3.0, linewidth=1.2, label='Cash EPS FY5'),
        'EPS_CashAct_coverage': dict(color=COLORS['green'], marker='D', markersize=3.4, linewidth=1.3, label='Cash EPS Act'),
    }

    for col in plot_cols:
        kwargs = line_style.get(col, dict(marker='o', markersize=3.0, linewidth=1.2, label=col))
        ax.plot(q_dates, quarterly_eps_cov[col], **kwargs)

    ax.set_title('Cash EPS Quarterly Coverage by Horizon')
    ax.set_xlabel('As-of quarter')
    ax.set_ylabel('Coverage (%)')
    ax.set_ylim(0, 100)

    style_axes(ax, grid_axis='y', grid_alpha=0.25)
    style_time_axis(
        ax,
        x_min=q_dates.min(),
        x_max=q_dates.max(),
        x_ticks=q_dates,
        date_fmt='%Y',
    )
    style_legend(ax, loc='lower left')

    plt.tight_layout()
    plt.show()
else:
    print('No Cash EPS coverage columns available for Step 3B plot.')

## 4. Long-Term Growth (LTG)

In [ ]:
import numpy as np
import pandas as pd

from standard_lseg_series_puller import (
    SeriesPullConfig,
    run_standard_pull,
)

# Step 4 — LTG Pull (ultra-lean notebook wrapper)
LTG_FIELD = 'TR.LTGMean'
LTG_COL = 'LTG'

cfg = SeriesPullConfig(
    field=LTG_FIELD,
    fallback_field=None,
    output_col=LTG_COL,
    intervals=('quarterly', 'monthly'),
    asof_tolerance_days=120,
    batch_size=10,
    batch_pause_sec=0,
    min_asof_date=pd.Timestamp('1998-12-31'),
    force_refresh=False,
    cache_only=False,
    skip_known_bad_ids=True,
    bad_id_cooldown_days=30,
)

ltg_source = pd.read_parquet(EURO500_EPS_PATH).copy()
# Pull selection must not depend on previously stored LTG values in euro500_macaulay.parquet.
ltg_source[LTG_COL] = np.nan

res = run_standard_pull(
    pull_type='series',
    source_df=ltg_source,
    config=cfg,
    cache_dir=CACHE_DATA_DIR / 'ltg_cache_by_company',
    step_rows_path=CACHE_DATA_DIR / 'ltg_rows.parquet',
    checkpoint_path=CACHE_DATA_DIR / 'ltg_checkpoint.parquet',
    bad_ids_path=CACHE_DATA_DIR / 'ltg_bad_ids.csv',
    skip_filled_primary=False,
    merge_back=True,
    diag_prefix='ltg_',
)

out = res['merged_df'].copy()
out.to_parquet(EURO500_EPS_PATH, index=False)

print('Run stats:', res['stats'])







Standard Series Pull Overview
series_specs: LTG<-['TR.LTGMean']
request_rows: 54,500
coverage: all_companies=1,163 | full_coverage=1 | partial_coverage=19 | bad_ids=10 | remaining=1,133
mode: CACHE+NETWORK | batch_size: 10
[BATCH 1/114] companies=10 idx=1-10
[BATCH 1/114] [1/1133] firm_id=FIRM0000040 | cand_used=4/4 | unresolved=51 | found_LTG=58 | range_in_index=1998-12-31:2025-12-31 | pulled_range=1999-06-30:2025-12-31 | tried_ids: ISIN:FR0014004L86 | ISIN:FR0000121725 | RIC:AM.PA | RIC:AVMD.PA
[BATCH 1/114] [2/1133] firm_id=FIRM0000041 | cand_used=3/3 | unresolved=14 | found_LTG=4 | range_in_index=1998-12-31:2006-03-31 | pulled_range=1999-09-30:2001-12-31 | tried_ids: ISIN:FR0000066052 | RIC:AVOM.PA | RIC:AVOM.LN
[BATCH 1/114] [3/1133] firm_id=FIRM0000042 | cand_used=2/2 | unresolved=5 | found_LTG=10 | range_in_index=1998-12-31:2002-06-28 | pulled_range=1998-12-31:2001-06-29 | tried_ids: ISIN:ES0112458312 | RIC:AZK.MC
[BATCH 1/114] [4/1133] firm_id=FIRM0000043 | cand_used=3/3 | unr

### 4A. Coverage-Analyse EPS und LTG

In [ ]:

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

cov = pd.read_parquet(EURO500_EPS_PATH).copy()

# Datumsanker fuer Jahresanalyse
if 'date' not in cov.columns:
    raise ValueError('Missing required date column for completeness analysis.')
cov['date'] = pd.to_datetime(cov['date'], errors='coerce').dt.normalize()

cov = cov[cov['date'].notna()].copy()
if cov.empty:
    raise ValueError('No valid date values found for completeness analysis.')

cov['year'] = cov['date'].dt.year

eps_cols = [c for c in ['EPS_FY1', 'EPS_FY2', 'EPS_FY3', 'EPS_FY4', 'EPS_FY5']]
for c in eps_cols + [LTG_COL]:
    if c not in cov.columns:
        cov[c] = np.nan
    cov[c] = pd.to_numeric(cov[c], errors='coerce')

cov['eps_any'] = cov[eps_cols].notna().any(axis=1)
cov['eps_all'] = cov[eps_cols].notna().all(axis=1)
cov['eps_partial'] = cov[['EPS_FY1', 'EPS_FY2', 'EPS_FY3']].notna().all(axis=1)
cov['eps_ltg_all'] = cov['eps_all'] & cov[LTG_COL].notna()
cov['eps_non_null_n'] = cov[eps_cols].notna().sum(axis=1)

agg = {'rows': ('year', 'size')}
for c in eps_cols + [LTG_COL]:
    agg[f'{c}_non_null'] = (c, lambda s: int(s.notna().sum()))
for c in ['eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
    agg[f'{c}_non_null'] = (c, lambda s: int(s.sum()))

by_year = cov.groupby('year', as_index=False).agg(**agg)

# Coverage-Raten ergaenzen
for c in eps_cols + [LTG_COL, 'eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
    by_year[f'{c}_coverage'] = by_year[f'{c}_non_null'] / by_year['rows']

# Lesbare Spaltenauswahl
show_cols = ['year', 'rows']
show_cols += [f'{c}_coverage' for c in eps_cols]
show_cols += [f'{LTG_COL}_coverage', 'eps_any_coverage', 'eps_all_coverage', 'eps_partial_coverage', 'eps_ltg_all_coverage']

print('Coverage by year (Step 3+4 outputs):')
display(by_year[show_cols].sort_values('year'))
print('Rows with partial Cash EPS coverage (FY1-FY3 complete):', int(cov['eps_partial'].sum()))

# Diagnostics: verify eps_any vs eps_all relationship
pattern = (
    cov.groupby('eps_non_null_n', as_index=False)
    .agg(rows=('eps_non_null_n', 'size'))
    .sort_values('eps_non_null_n')
)
pattern['share'] = pattern['rows'] / len(cov)
print('EPS non-null count distribution (0..5):')
display(pattern)
print('Rows with eps_any != eps_all:', int((cov['eps_any'] != cov['eps_all']).sum()))
print('Rows with eps_all=True and eps_any=False (should be 0):', int(((cov['eps_all']) & (~cov['eps_any'])).sum()))

# Schwachstellen: schlechteste Jahre
worst = (
    by_year[['year', 'eps_ltg_all_coverage']]
    .sort_values('eps_ltg_all_coverage', ascending=True)
    .head(10)
)
print('10 years with lowest joint EPS+LTG completeness:')
display(worst)

# Plot: Zeitverlauf der wichtigsten Coverage-Metriken
plot_cols = [
    'eps_any_coverage',
    'eps_all_coverage',
    'eps_partial_coverage',
    f'{LTG_COL}_coverage',
]

fig, ax = plt.subplots(figsize=(12, 5))
legend_labels = {
    'eps_any_coverage': 'Cash EPS Any (FY1-FY5)',
    'eps_all_coverage': 'Cash EPS All (FY1-FY5)',
    'eps_partial_coverage': 'Cash EPS Partial (FY1-FY3)',
    f'{LTG_COL}_coverage': 'LTG',
}
for col in plot_cols:
    ax.plot(by_year['year'], by_year[col], marker='o', label=legend_labels.get(col, col))

ax.set_title('Coverage Over Time: Cash EPS and LTG')
ax.set_xlabel('Year')
ax.set_ylabel('Coverage Share')
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### 4B. LTG-Qualitaetscheck (Plausibilitaet)

In [ ]:
LTG_COL = 'LTG'

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

ltg_df = pd.read_parquet(EURO500_EPS_PATH).copy()
if LTG_COL not in ltg_df.columns:
    raise KeyError(f'Column {LTG_COL} not found. Run Step 4 first.')

# Datentypen / Datumsfelder
for dc in ['date', 'date', 'formation_date', 'effective_date']:
    if dc in ltg_df.columns:
        ltg_df[dc] = pd.to_datetime(ltg_df[dc], errors='coerce')

ltg_df[LTG_COL] = pd.to_numeric(ltg_df[LTG_COL], errors='coerce')

if 'date' in ltg_df.columns and ltg_df['date'].notna().any():
    ltg_df['year'] = ltg_df['date'].dt.year
elif 'date' in ltg_df.columns:
    ltg_df['year'] = ltg_df['date'].dt.year
else:
    ltg_df['year'] = pd.NA

# 1) Overall coverage + basic stats
n_total = len(ltg_df)
n_non_null = int(ltg_df[LTG_COL].notna().sum())
n_missing = n_total - n_non_null
coverage = (n_non_null / n_total) if n_total else np.nan

outlier_lo = (ltg_df[LTG_COL] < -1).sum()
outlier_hi = (ltg_df[LTG_COL] > 1).sum()

summary = pd.DataFrame([{
    'rows_total': n_total,
    'ltg_non_null': n_non_null,
    'ltg_missing': n_missing,
    'coverage_share': coverage,
    'ltg_lt_-1_count': int(outlier_lo),
    'ltg_gt_+1_count': int(outlier_hi),
}])
display(summary)

ltg_ok = ltg_df[LTG_COL].dropna()
if len(ltg_ok) > 0:
    print('LTG describe (raw):')
    display(ltg_ok.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    p01, p99 = ltg_ok.quantile([0.01, 0.99])
    ltg_w = ltg_ok.clip(lower=p01, upper=p99)
    print('LTG describe (winsorized 1%-99%):')
    display(ltg_w.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(ltg_ok, bins=80)
    ax[0].set_title('LTG distribution (raw)')
    ax[0].set_xlabel(LTG_COL)

    ax[1].hist(ltg_w, bins=80)
    ax[1].set_title('LTG distribution (winsor 1%-99%)')
    ax[1].set_xlabel(LTG_COL)
    plt.tight_layout()
    plt.show()


    # Zusatzplot: LTG coverage by year (Non-Financials only)
    if 'trbc_sector' in ltg_df.columns:
        ltg_nonfin = ltg_df[ltg_df['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if ltg_nonfin['year'].notna().any():
            cov_year_nf = ltg_nonfin.groupby('year', as_index=False).agg(
                rows=(LTG_COL, 'size'),
                non_null=(LTG_COL, lambda s: s.notna().sum()),
                median=(LTG_COL, 'median'),
                p10=(LTG_COL, lambda s: s.quantile(0.10)),
                p90=(LTG_COL, lambda s: s.quantile(0.90)),
            )
            cov_year_nf['coverage'] = cov_year_nf['non_null'] / cov_year_nf['rows']

            fig, ax = plt.subplots(1, 2, figsize=(12, 4))
            ax[0].plot(cov_year_nf['year'], cov_year_nf['coverage'])
            ax[0].set_title('LTG coverage by year (Non-Financials)')
            ax[0].set_xlabel('year')
            ax[0].set_ylabel('coverage')
            ax[0].set_ylim(0, 1.02)
            ax[0].grid(alpha=0.3)

            ax[1].plot(cov_year_nf['year'], cov_year_nf['median'], label='median')
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p10'], label='p10', alpha=0.7)
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p90'], label='p90', alpha=0.7)
            ax[1].set_title('LTG level by year (Non-Financials)')
            ax[1].set_xlabel('year')
            ax[1].legend()
            ax[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print('No valid year info for LTG Non-Financials coverage plot.')
    else:
        print('Column trbc_sector missing; skipping LTG Non-Financials coverage plot.')


else:
    print('No non-null LTG values found.')

# 2) Coverage over time
if 'year' in ltg_df.columns and ltg_df['year'].notna().any():
    cov_year = ltg_df.groupby('year', as_index=False).agg(
        rows=(LTG_COL, 'size'),
        non_null=(LTG_COL, lambda s: s.notna().sum()),
        median=(LTG_COL, 'median'),
        p10=(LTG_COL, lambda s: s.quantile(0.10)),
        p90=(LTG_COL, lambda s: s.quantile(0.90)),
    )
    cov_year['coverage'] = cov_year['non_null'] / cov_year['rows']
    display(cov_year.head(20))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(cov_year['year'], cov_year['coverage'])
    ax[0].set_title('LTG coverage by year')
    ax[0].set_xlabel('year')
    ax[0].set_ylabel('coverage')
    ax[0].grid(alpha=0.3)

    ax[1].plot(cov_year['year'], cov_year['median'], label='median')
    ax[1].plot(cov_year['year'], cov_year['p10'], label='p10', alpha=0.7)
    ax[1].plot(cov_year['year'], cov_year['p90'], label='p90', alpha=0.7)
    ax[1].set_title('LTG level by year')
    ax[1].set_xlabel('year')
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()



    # Zusatzplot: LTG coverage by year (Non-Financials, coverage section)
    if 'trbc_sector' in ltg_df.columns:
        ltg_nonfin_cov = ltg_df[ltg_df['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if ltg_nonfin_cov['year'].notna().any():
            cov_year_nf = ltg_nonfin_cov.groupby('year', as_index=False).agg(
                rows=(LTG_COL, 'size'),
                non_null=(LTG_COL, lambda s: s.notna().sum()),
                median=(LTG_COL, 'median'),
                p10=(LTG_COL, lambda s: s.quantile(0.10)),
                p90=(LTG_COL, lambda s: s.quantile(0.90)),
            )
            cov_year_nf['coverage'] = cov_year_nf['non_null'] / cov_year_nf['rows']

            fig, ax = plt.subplots(1, 2, figsize=(12, 4))
            ax[0].plot(cov_year_nf['year'], cov_year_nf['coverage'])
            ax[0].set_title('LTG coverage by year (Non-Financials)')
            ax[0].set_xlabel('year')
            ax[0].set_ylabel('coverage')
            ax[0].set_ylim(0, 1.02)
            ax[0].grid(alpha=0.3)

            ax[1].plot(cov_year_nf['year'], cov_year_nf['median'], label='median')
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p10'], label='p10', alpha=0.7)
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p90'], label='p90', alpha=0.7)
            ax[1].set_title('LTG level by year (Non-Financials)')
            ax[1].set_xlabel('year')
            ax[1].legend()
            ax[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print('No valid year info for LTG Non-Financials coverage plot.')
    else:
        print('Column trbc_sector missing; skipping LTG Non-Financials coverage plot.')

# 3) Cross-section by sector / country
for grp_col in ['trbc_sector', 'hq_country']:
    if grp_col in ltg_df.columns:
        g = (
            ltg_df.groupby(grp_col, dropna=False)[LTG_COL]
            .agg(n='size', non_null=lambda s: s.notna().sum(), median='median', mean='mean')
            .reset_index()
        )
        g['coverage'] = g['non_null'] / g['n']
        g = g.sort_values(['coverage', 'n'], ascending=[False, False])
        print(f'By {grp_col}:')
        display(g.head(20))





## 5. Risk free rate

In [ ]:
# Step 5 — Market risk-free rate merge (quarter-start as-of, per firm)
# Strict requirement: use annualized €STR only (rf_estr_annual).
RF_SOURCE_PRIMARY = DATA_DIR / 'euro500_index_returns.parquet'
RF_CACHE_DIR = CACHE_DATA_DIR / 'rf_cache'
MACAULAY_PATH = DATA_DIR / 'euro500_macaulay.parquet'
TARGET_PATH = MACAULAY_PATH

if not TARGET_PATH.exists():
    raise FileNotFoundError(f'Missing target file: {TARGET_PATH}')

tgt = pd.read_parquet(TARGET_PATH).copy()
if 'date' not in tgt.columns:
    raise KeyError("'date' column missing in target macaulay table")

rf_series = None
rf_origin = None

# 1) Preferred source: euro500_index_returns.parquet with rf_estr_annual
if RF_SOURCE_PRIMARY.exists():
    src = pd.read_parquet(RF_SOURCE_PRIMARY).copy()
    if 'date' in src.columns and 'rf_estr_annual' in src.columns:
        tmp = src[['date', 'rf_estr_annual']].copy()
        tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce').dt.normalize()
        tmp['rf_estr_annual'] = pd.to_numeric(tmp['rf_estr_annual'], errors='coerce')
        tmp = tmp[tmp['date'].notna() & tmp['rf_estr_annual'].notna()].sort_values('date').drop_duplicates('date', keep='last')
        if len(tmp):
            rf_series = tmp
            rf_origin = str(RF_SOURCE_PRIMARY)

# 2) Fallback: newest RF cache file from Euro500_IndexReturns Step 2
if rf_series is None and RF_CACHE_DIR.exists():
    cache_files = sorted(RF_CACHE_DIR.glob('rf_estr_v2_*.parquet'))
    if cache_files:
        cache = pd.read_parquet(cache_files[-1]).copy()
        if 'date' in cache.columns and 'rf_estr_annual' in cache.columns:
            tmp = cache[['date', 'rf_estr_annual']].copy()
            tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce').dt.normalize()
            tmp['rf_estr_annual'] = pd.to_numeric(tmp['rf_estr_annual'], errors='coerce')
            tmp = tmp[tmp['date'].notna() & tmp['rf_estr_annual'].notna()].sort_values('date').drop_duplicates('date', keep='last')
            if len(tmp):
                rf_series = tmp
                rf_origin = str(cache_files[-1])

if rf_series is None:
    raise KeyError(
        "rf_estr_annual not found. Run Euro500_IndexReturns Step 2 first and ensure "
        "rf_estr_annual is saved in euro500_index_returns.parquet or rf_cache."
    )

# Requested key: for each firm row, use quarter-start and attach current market annual RF rate.
tgt['date'] = pd.to_datetime(tgt['date'], errors='coerce').dt.normalize()
tgt['quarter_start_date'] = tgt['date'].dt.to_period('Q').dt.start_time.dt.normalize()

left = tgt.sort_values('quarter_start_date').copy()
right = rf_series.rename(columns={'date': 'rf_obs_date', 'rf_estr_annual': 'market_risk_free_rate_annual'}).sort_values('rf_obs_date')

left = pd.merge_asof(
    left,
    right,
    left_on='quarter_start_date',
    right_on='rf_obs_date',
    direction='backward',
)

out = left.sort_index().copy()
out['market_risk_free_rate_annual'] = pd.to_numeric(out['market_risk_free_rate_annual'], errors='coerce')

# Explicitly remove any daily RF carry-over columns from macaulay output.
for c in ['rf_daily', 'market_risk_free_rate', 'daily_rf']:
    if c in out.columns:
        out = out.drop(columns=[c])

# Persist only the unified macaulay table (single-output design).
out.to_parquet(MACAULAY_PATH, index=False)

print('RF origin:', rf_origin)
print('RF column used: rf_estr_annual')
print('Target input:', TARGET_PATH)
print('Rows:', len(out), '| annual rf non-null share:', float(out['market_risk_free_rate_annual'].notna().mean()))